# PARSHO — interactive single-image analysis

Upload either one multichannel microscopy image or several aligned channel images, inspect every intermediate result, segment cells with Cellpose, detect nuclei and aggregates, run shape-adapted radial analysis, and download a ZIP containing all figures, masks, settings, and CSV tables.

Run cells from top to bottom. Form values can be changed before running a cell. This notebook intentionally processes one field of view at a time. A Colab GPU runtime is recommended: **Runtime → Change runtime type → T4 GPU**.

## 1. Install PARSHO
The runtime restarts are not normally required after this installation.

In [ ]:
#@title Install package and image readers
PARSHO_SOURCE = "git+https://github.com/Fraternalilab/PARSHO.git" #@param {type:"string"}
%pip install -q "parsho[microscopy-io] @ ${PARSHO_SOURCE}"


In [ ]:
#@title Import libraries and initialize output folder
from pathlib import Path
import json, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tifffile
from IPython.display import display, Image as DisplayImage
from cellpose import core, models
import cellpose.utils as cellpose_utils
from parsho import extract_channels
from parsho.distribution import compute_nucleus_centered_distribution, nucleus_centroids_by_cell
from parsho.maskfilters import build_overlay, compute_cell_metrics, filter_cells_by_overlap, mask_overlay_to_transfected, subtract_nuclear_from_aggregate
from parsho.plotting import plot_aggregate_channel, plot_aggregate_channel_color, plot_aggregate_channel_color_labelled, plot_nucleus_centered_distribution, plot_segmentation_result
from parsho.segmentation import extract_masks, find_optimal_threshold, re_threshold_masks
from parsho.utils import radial_distributions_to_records
OUTPUT_DIR = Path('/content/parsho_results')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
GPU = core.use_gpu()
print(f'Cellpose GPU available: {GPU}')
print(f'Results folder: {OUTPUT_DIR}')
def show_saved(filename):
    display(DisplayImage(filename=str(OUTPUT_DIR / filename)))

## 2. Upload and assign images
Upload the sample and, if desired, positive/negative controls together. ND2, LIF, LOF, CZI, TIFF/OME-TIFF, DICOM, JPEG, BMP, and PNG are accepted. Large files can instead be copied to Google Drive and referenced by their mounted path.

In [ ]:
#@title Upload image files
from google.colab import files
uploaded = files.upload()
print('Uploaded:', list(uploaded))

In [ ]:
#@title Configure the sample image(s)
INPUT_MODE = "one multichannel image" #@param ["one multichannel image", "separate channel images"]
MULTICHANNEL_FILE = "sample.ome.tif" #@param {type:"string"}
CELL_FILES = "cell.tif" #@param {type:"string"}
AGGREGATE_FILE = "aggregate.tif" #@param {type:"string"}
NUCLEUS_FILE = "nucleus.tif" #@param {type:"string"}
SEGMENTATION_CHANNELS = "0" #@param {type:"string"}
AGGREGATE_CHANNEL = 1 #@param {type:"integer"}
NUCLEUS_CHANNEL = 2 #@param {type:"integer"}
SCENE = 0 #@param {type:"integer"}
Z_HANDLING = "maximum projection" #@param ["maximum projection", "single plane"]
Z_PLANE = 0 #@param {type:"integer"}

def csv_items(value):
    return [item.strip() for item in value.split(',') if item.strip()]

def to_2d(channel, z_handling=Z_HANDLING, z_plane=Z_PLANE):
    array = np.squeeze(np.asarray(channel))
    if array.ndim == 2:
        return array
    if array.ndim != 3:
        raise ValueError(f'Each selected channel must reduce to ZYX or YX; got {array.shape}')
    if z_handling == 'maximum projection':
        return array.max(axis=0)
    if not 0 <= z_plane < array.shape[0]:
        raise IndexError(f'Z_PLANE {z_plane} is outside 0..{array.shape[0] - 1}')
    return array[z_plane]

def channels_from_file(filename, scene=0):
    path = Path('/content') / filename if not Path(filename).is_absolute() else Path(filename)
    return [to_2d(channel) for channel in extract_channels(path, normalize=False, scene=scene)]

if INPUT_MODE == 'one multichannel image':
    sample_channels = channels_from_file(MULTICHANNEL_FILE, SCENE)
    segmentation_indices = [int(value) for value in csv_items(SEGMENTATION_CHANNELS)]
    segmentation_channels = [sample_channels[index] for index in segmentation_indices]
    aggregate_image = sample_channels[AGGREGATE_CHANNEL]
    nucleus_image = sample_channels[NUCLEUS_CHANNEL]
    sample_name = Path(MULTICHANNEL_FILE).stem
else:
    segmentation_channels = [channels_from_file(name, SCENE)[0] for name in csv_items(CELL_FILES)]
    aggregate_image = channels_from_file(AGGREGATE_FILE, SCENE)[0]
    nucleus_image = channels_from_file(NUCLEUS_FILE, SCENE)[0]
    sample_name = Path(AGGREGATE_FILE).stem

shapes = [image.shape for image in [*segmentation_channels, aggregate_image, nucleus_image]]
if len(set(shapes)) != 1:
    raise ValueError(f'All selected channels must be aligned; received shapes {shapes}')
print(f'Loaded {len(segmentation_channels)} segmentation channel(s), aggregate and nucleus: {shapes[0]}')

In [ ]:
#@title Inspect selected channels
images = [*segmentation_channels, aggregate_image, nucleus_image]
titles = [f'Segmentation {i}' for i in range(len(segmentation_channels))] + ['Aggregates', 'Nucleus']
fig, axes = plt.subplots(1, len(images), figsize=(5 * len(images), 5), squeeze=False)
for axis, image, title in zip(axes[0], images, titles):
    axis.imshow(image, cmap='gray')
    axis.set_title(f'{title}\nshape={image.shape}, dtype={image.dtype}')
    axis.axis('off')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / '01_input_channels.png', dpi=200, bbox_inches='tight')
plt.show()

## 3. Optional threshold controls
Controls calibrate a fixed aggregate threshold exactly as in the package tutorial. Each control may be one multichannel image. If controls are disabled, choose Otsu or a manual threshold later.

In [ ]:
#@title Configure optional controls
USE_CONTROLS = False #@param {type:"boolean"}
POSITIVE_CONTROL_FILE = "positive_control.ome.tif" #@param {type:"string"}
NEGATIVE_CONTROL_FILE = "negative_control.ome.tif" #@param {type:"string"}
CONTROL_SEGMENTATION_CHANNELS = "0" #@param {type:"string"}
CONTROL_AGGREGATE_CHANNEL = 1 #@param {type:"integer"}
CONTROL_NUCLEUS_CHANNEL = 2 #@param {type:"integer"}
if USE_CONTROLS:
    def load_control(filename):
        channels = channels_from_file(filename, SCENE)
        indices = [int(value) for value in csv_items(CONTROL_SEGMENTATION_CHANNELS)]
        return [channels[i] for i in indices], channels[CONTROL_AGGREGATE_CHANNEL], channels[CONTROL_NUCLEUS_CHANNEL]
    positive_control = load_control(POSITIVE_CONTROL_FILE)
    negative_control = load_control(NEGATIVE_CONTROL_FILE)
    print('Controls loaded. They will be segmented with the settings in the next cell.')
else:
    positive_control = negative_control = None
    print('Control calibration disabled.')

## 4. Cellpose segmentation
All selected segmentation channels are independently percentile-normalized and combined into a multichannel image. Adjust the form and rerun until the displayed outlines are satisfactory.

In [ ]:
#@title Run Cellpose (rerun after changing parameters)
CELLPOSE_MODEL = "cpsam_v2" #@param {type:"string"}
SEGMENTATION_COMBINATION = "mean projection" #@param ["mean projection", "maximum projection", "multichannel stack"]
DIAMETER = 0 #@param {type:"number"}
FLOW_THRESHOLD = 0.0 #@param {type:"number"}
CELLPROB_THRESHOLD = -2.0 #@param {type:"number"}
MIN_CELL_SIZE = 15 #@param {type:"integer"}
BATCH_SIZE = 32 #@param {type:"integer"}

def percentile_normalize(image):
    image = image.astype(np.float32)
    low, high = np.percentile(image, (1, 99))
    return np.zeros_like(image) if high <= low else np.clip((image - low) / (high - low), 0, 1)

def combine_for_cellpose(channels):
    normalized = [percentile_normalize(channel) for channel in channels]
    if len(normalized) == 1:
        return normalized[0]
    stack = np.stack(normalized, axis=0)
    if SEGMENTATION_COMBINATION == 'mean projection':
        return stack.mean(axis=0)
    if SEGMENTATION_COMBINATION == 'maximum projection':
        return stack.max(axis=0)
    if len(normalized) > 3:
        raise ValueError('Cellpose multichannel input supports at most 3 channels; use a projection to combine more.')
    return np.moveaxis(stack, 0, -1)

cellpose_parameters = dict(
    batch_size=BATCH_SIZE, diameter=None if DIAMETER <= 0 else DIAMETER,
    flow_threshold=FLOW_THRESHOLD, cellprob_threshold=CELLPROB_THRESHOLD,
    min_size=MIN_CELL_SIZE, normalize=False,
)
model = models.CellposeModel(gpu=GPU, pretrained_model=CELLPOSE_MODEL)
segmentation_input = combine_for_cellpose(segmentation_channels)
masks, flows, _ = model.eval(segmentation_input, channel_axis=-1 if segmentation_input.ndim == 3 else None, **cellpose_parameters)
print(f'Detected {int(masks.max())} cells')
plot_segmentation_result(segmentation_input, masks, flows, save_path=OUTPUT_DIR / '02_cellpose_segmentation.png')
show_saved('02_cellpose_segmentation.png')
tifffile.imwrite(OUTPUT_DIR / 'cell_masks.tif', masks.astype(np.int32))

In [ ]:
#@title Calibrate aggregate threshold from controls (optional)
CONTROL_AGGREGATE_MIN_SIZE = 2 #@param {type:"integer"}
CONTROL_NUCLEUS_MIN_SIZE = 5 #@param {type:"integer"}
calibrated_threshold = None
if USE_CONTROLS:
    control_results = []
    for name, (seg_channels, agg, nuc) in [('positive', positive_control), ('negative', negative_control)]:
        cp_input = combine_for_cellpose(seg_channels)
        cp_masks, cp_flows, _ = model.eval(cp_input, channel_axis=-1 if cp_input.ndim == 3 else None, **cellpose_parameters)
        agg_binary, _, agg_threshold = extract_masks(agg, cp_masks, method='otsu', min_size_px=CONTROL_AGGREGATE_MIN_SIZE)
        nuc_binary, _, _ = extract_masks(nuc, cp_masks, method='otsu', min_size_px=CONTROL_NUCLEUS_MIN_SIZE)
        control_results.append((agg, cp_masks, nuc_binary, agg_threshold))
        plot_segmentation_result(cp_input, cp_masks, cp_flows, save_path=OUTPUT_DIR / f'control_{name}_segmentation.png')
        show_saved(f'control_{name}_segmentation.png')
        print(f'{name.title()} control Otsu threshold: {agg_threshold}')
    pos_agg, pos_masks, pos_nuc, pos_threshold = control_results[0]
    neg_agg, neg_masks, neg_nuc, neg_threshold = control_results[1]
    calibrated_threshold, positive_pixels = find_optimal_threshold(pos_agg, pos_masks, neg_agg, neg_masks, pos_nuc, neg_nuc, t_min=min(neg_threshold, pos_threshold), t_max=max(neg_threshold, pos_threshold))
    print(f'Calibrated threshold: {calibrated_threshold}; retained positive pixels: {positive_pixels}')
else:
    print('Skipped: controls are disabled.')

## 5. Detect nuclei and aggregates
Nuclei use Otsu thresholding. Aggregates may use the controls, their own Otsu threshold, or a fixed manual threshold. Nuclear pixels can optionally be removed from the aggregate mask and intensity image.

In [ ]:
#@title Detect and inspect nuclei and aggregates
AGGREGATE_THRESHOLD_MODE = "Otsu" #@param ["controls", "Otsu", "manual"]
MANUAL_AGGREGATE_THRESHOLD = 1000.0 #@param {type:"number"}
AGGREGATE_MIN_SIZE = 2 #@param {type:"integer"}
NUCLEUS_MIN_SIZE = 5 #@param {type:"integer"}
REMOVE_NUCLEAR_SIGNAL_FROM_AGGREGATES = True #@param {type:"boolean"}

nucleus_binary, nucleus_labels, nucleus_threshold = extract_masks(nucleus_image, masks, method='otsu', min_size_px=NUCLEUS_MIN_SIZE)
if AGGREGATE_THRESHOLD_MODE == 'controls':
    if calibrated_threshold is None:
        raise ValueError('Enable controls and run the calibration cell first.')
    aggregate_threshold = calibrated_threshold
    aggregate_binary, aggregate_labels = re_threshold_masks(aggregate_image, masks, min_size_px=AGGREGATE_MIN_SIZE, thresh=aggregate_threshold)
elif AGGREGATE_THRESHOLD_MODE == 'manual':
    aggregate_threshold = MANUAL_AGGREGATE_THRESHOLD
    aggregate_binary, aggregate_labels = re_threshold_masks(aggregate_image, masks, min_size_px=AGGREGATE_MIN_SIZE, thresh=aggregate_threshold)
else:
    aggregate_binary, aggregate_labels, aggregate_threshold = extract_masks(aggregate_image, masks, method='otsu', min_size_px=AGGREGATE_MIN_SIZE)
if REMOVE_NUCLEAR_SIGNAL_FROM_AGGREGATES:
    aggregate_binary, aggregate_labels = subtract_nuclear_from_aggregate(aggregate_binary, aggregate_labels, nucleus_binary)
print(f'Nucleus threshold: {nucleus_threshold}; aggregate threshold: {aggregate_threshold}')
print('Nucleus mask')
plot_aggregate_channel(nucleus_binary, save_path=OUTPUT_DIR / '03_nucleus_mask.png')
show_saved('03_nucleus_mask.png')
print('Aggregate mask')
plot_aggregate_channel(aggregate_binary, save_path=OUTPUT_DIR / '04_aggregate_mask.png')
show_saved('04_aggregate_mask.png')
tifffile.imwrite(OUTPUT_DIR / 'nucleus_mask.tif', nucleus_binary.astype(np.uint8))
tifffile.imwrite(OUTPUT_DIR / 'aggregate_mask.tif', aggregate_binary.astype(np.uint8))

In [ ]:
#@title Filter cells and inspect the categorical overlay
FILTER_CELLS_WITH_NUCLEUS = True #@param {type:"boolean"}
FILTER_CELLS_WITH_AGGREGATES = False #@param {type:"boolean"}
all_labels = [int(label) for label in np.unique(masks) if label != 0]
overlap_masks = []
if FILTER_CELLS_WITH_NUCLEUS:
    overlap_masks.append(nucleus_labels)
if FILTER_CELLS_WITH_AGGREGATES:
    overlap_masks.append(aggregate_labels)
retained_labels = (
    filter_cells_by_overlap(masks, *overlap_masks)
    if overlap_masks
    else all_labels
)
outlines = cellpose_utils.masks_to_outlines(masks)
overlay = build_overlay(masks, nucleus_labels, aggregate_labels, outlines)
filtered_overlay = mask_overlay_to_transfected(overlay, masks, retained_labels)
display_numbers = {label: index for index, label in enumerate(retained_labels)}
print(f'Retained {len(retained_labels)} of {len(all_labels)} cells')
plot_aggregate_channel_color(overlay, save_path=OUTPUT_DIR / '05_all_cells_overlay.png')
show_saved('05_all_cells_overlay.png')
plot_aggregate_channel_color_labelled(filtered_overlay, masks, display_numbers, save_path=OUTPUT_DIR / '06_retained_cells.png')
show_saved('06_retained_cells.png')

## 6. Cell measurements and radial analysis
Radial fields are centered on each detected nucleus and adapt to the cell boundary. Cells without a usable nuclear centroid are omitted.

In [ ]:
#@title Calculate measurements and radial distributions
RADIAL_BINS = 10 #@param {type:"integer"}
ZERO_NUCLEAR_INTENSITY_FOR_RADIAL_ANALYSIS = True #@param {type:"boolean"}
metrics = compute_cell_metrics(masks, nucleus_labels, aggregate_labels, retained_labels)
cell_records = []
for item in metrics:
    cell_records.append({'sample': sample_name, 'cell_number': display_numbers[item.label], 'cell_label': item.label, 'cell_area': item.cell_area, 'aggregate_area': item.agg_area, 'nucleus_area': item.nuc_area, 'jaccard': item.jaccard, 'cell_aspect_ratio': item.cell_aspect_ratio, 'cell_circularity': item.cell_circularity, 'aggregate_aspect_ratio': item.agg_aspect_ratio, 'aggregate_circularity': item.agg_circularity})
cell_df = pd.DataFrame(cell_records)
radial_intensity = aggregate_image.copy()
if ZERO_NUCLEAR_INTENSITY_FOR_RADIAL_ANALYSIS:
    radial_intensity[nucleus_binary != 0] = 0
nucleus_centroids = nucleus_centroids_by_cell(masks, nucleus_binary, cell_labels=retained_labels)
distributions = compute_nucleus_centered_distribution(masks, nucleus_centroids, aggregate_binary, radial_intensity, radial_bins=RADIAL_BINS, cell_labels=list(nucleus_centroids))
radial_records = radial_distributions_to_records({sample_name: distributions})
for record in radial_records:
    record['cell_label'] = display_numbers[record['cell_label']]
radial_df = pd.DataFrame(radial_records)
cell_df.to_csv(OUTPUT_DIR / 'cell_measurements.csv', index=False)
radial_df.to_csv(OUTPUT_DIR / 'radial_distribution.csv', index=False)
print('Cell measurements')
display(cell_df)
print('Radial measurements')
display(radial_df)

In [ ]:
#@title Show and save radial diagnostics
SHOW_CELL_NUMBER = 0 #@param {type:"integer"}
for internal_label, distribution in distributions.items():
    cell_number = display_numbers[internal_label]
    plot_nucleus_centered_distribution(masks, aggregate_binary, radial_intensity, distribution, save_path=OUTPUT_DIR / f'07_radial_cell_{cell_number}.png', show=cell_number == SHOW_CELL_NUMBER, cell_name=f'Cell {cell_number}')
print(f'Saved {len(distributions)} radial diagnostic figures.')
if SHOW_CELL_NUMBER not in display_numbers.values():
    print('Requested cell number was not available; see the retained-cell overlay for valid numbers.')

## 7. Save and download everything

In [ ]:
#@title Create and download results ZIP
settings = {
    'sample_name': sample_name, 'input_mode': INPUT_MODE, 'scene': SCENE,
    'z_handling': Z_HANDLING, 'cellpose_model': CELLPOSE_MODEL,
    'segmentation_combination': SEGMENTATION_COMBINATION,
    'cellpose_parameters': cellpose_parameters,
    'aggregate_threshold_mode': AGGREGATE_THRESHOLD_MODE,
    'aggregate_threshold': float(aggregate_threshold),
    'nucleus_threshold': float(nucleus_threshold),
    'remove_nuclear_signal': REMOVE_NUCLEAR_SIGNAL_FROM_AGGREGATES,
    'filter_cells_with_nucleus': FILTER_CELLS_WITH_NUCLEUS,
    'filter_cells_with_aggregates': FILTER_CELLS_WITH_AGGREGATES,
    'radial_bins': RADIAL_BINS,
}
with open(OUTPUT_DIR / 'analysis_settings.json', 'w') as stream:
    json.dump(settings, stream, indent=2)
archive = shutil.make_archive('/content/PARSHO_results', 'zip', OUTPUT_DIR)
print(f'Created {archive}')
files.download(archive)

## Reading the output
- `cell_measurements.csv`: one row per retained cell with morphology and overlap measurements.
- `radial_distribution.csv`: one row per cell and radial bin with aggregate occupancy and intensity.
- `cell_masks.tif`, `nucleus_mask.tif`, `aggregate_mask.tif`: reusable raw masks.
- numbered PNG files: input, segmentation, detection, filtering, and per-cell radial quality control.
- `analysis_settings.json`: parameters required to reproduce the run.